# 第2讲：交通数据结构与Python分析工作流（学生操练）

        > 课程：《交通大数据分析与应用》  
        > 数据：课程模拟数据，不是实际监测数据  
        > 建议用时：18分钟

        ## 目标

        1. 修改DETECTOR_IDS并比较两个方向的数组聚合结果；
2. 按detector_id连接路段元数据，核对连接前后的行数；
3. 把15分钟数据重采样为小时数据并解释聚合函数；

        代码可以直接运行；请按`TODO`修改参数、核对输出并完成解释。

## 1. Setup｜环境、路径与参数

In [ ]:
from __future__ import annotations

from pathlib import Path
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
plt.rcParams.update({"figure.dpi": 120, "axes.grid": True, "grid.alpha": 0.25})
RANDOM_STATE = 42
print("环境已就绪；数据目录：", DATA_DIR.resolve())

## 2. 读取数据并构造二维数组

In [ ]:
traffic = pd.read_csv(DATA_DIR / "traffic_15min.csv", parse_dates=["timestamp"])
DETECTOR_IDS = ["D01", "D02", "D03"]  # TODO：改成另外三个检测器再比较
sample = traffic[traffic["detector_id"].isin(DETECTOR_IDS)].copy()
flow_matrix = sample.pivot(index="timestamp", columns="detector_id", values="flow_15min").sort_index()
values = flow_matrix.to_numpy()
print("DataFrame shape:", flow_matrix.shape, "NumPy shape:", values.shape)
display(flow_matrix.head())

## 3. 由Python核对axis方向

In [ ]:
axis0 = pd.Series(values.mean(axis=0), index=flow_matrix.columns, name="mean_flow_axis0")
axis1 = pd.Series(values.mean(axis=1), index=flow_matrix.index, name="mean_flow_axis1")
print("axis=0：压缩行，保留每个检测器；结果长度", len(axis0))
display(axis0.to_frame())
print("axis=1：压缩列，保留每个时刻；前5个结果")
display(axis1.head().to_frame())

## 4. 分组、连接与重采样

In [ ]:
meta = pd.read_csv(DATA_DIR / "segment_metadata.csv")
before = len(sample)
joined = sample.merge(meta, on=["detector_id", "segment_id", "direction", "road_type", "lanes"], how="left", validate="many_to_one")
assert len(joined) == before
summary = joined.groupby(["detector_id", "road_name"], as_index=False).agg(
    mean_speed=("speed_kmh", "mean"), total_flow=("flow_15min", "sum"))
display(summary.round(2))
hourly = (sample.set_index("timestamp").groupby("detector_id")["flow_15min"].resample("1h").sum().reset_index())
display(hourly.head())

## 5. 绘图并自检

In [ ]:
hourly_profile = hourly.assign(hour=hourly["timestamp"].dt.hour).groupby(["hour", "detector_id"])["flow_15min"].mean().unstack()
hourly_profile.plot(title="Mean hourly flow", linewidth=2)
plt.ylabel("vehicles/hour"); plt.tight_layout()
plt.savefig(OUTPUT_DIR / "lesson02_hourly_profile.png"); plt.show()
checks = {"axis0长度等于列数": len(axis0) == flow_matrix.shape[1], "axis1长度等于行数": len(axis1) == flow_matrix.shape[0], "连接行数不变": len(joined) == before}
status = "PASS" if all(checks.values()) else "CHECK"
(OUTPUT_DIR / "自检结果.txt").write_text(status, encoding="utf-8")
print(status, checks)

## Checks｜当堂记录

        - 修改DETECTOR_IDS并比较两个方向的数组聚合结果
- 按detector_id连接路段元数据，核对连接前后的行数
- 把15分钟数据重采样为小时数据并解释聚合函数

        **预期结果：** 数组方向核对表、按路段汇总表、小时流量曲线。

        **完成标准：** axis解释与Python输出一致；连接不增加或丢失观测；Notebook生成PASS。

        请在课堂记录中写下：改了什么参数、结果发生了什么变化、这个变化在交通问题中意味着什么。